## MODULE 2 — Series: Deep Dive
---
A **Series** is a 1-D labeled array. Every DataFrame column is a Series.

In [19]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [26]:
# ── 2.1 Creating Series ───────────────────────────────────────────────────────
# From list
revenue = pd.Series([120_000, 95_000, 142_000, 88_000, 175_000],
                   index =['Jan', 'Feb', 'Mar','Apr', 'May'],
                   name='Monthly Revenue')
print(revenue)
print(f'\nDtype:{revenue.dtype}')
print(f'Index:{revenue.index.tolist()}')

Jan    120000
Feb     95000
Mar    142000
Apr     88000
May    175000
Name: Monthly Revenue, dtype: int64

Dtype:int64
Index:['Jan', 'Feb', 'Mar', 'Apr', 'May']


In [27]:
# From dict — most common in real work
kpis = pd.Series({
    'Total Revenue': 620_000,
    'Total Orders' : 4_820,
    'Avg Order Value': 128.63,
    'Churn Rate %'  : 5.2,
    'NPS Score'     : 72
})
print(kpis)



Total Revenue      620000.00
Total Orders         4820.00
Avg Order Value       128.63
Churn Rate %            5.20
NPS Score              72.00
dtype: float64


In [34]:

# ── 2.2 Indexing & Selection ──────────────────────────────────────────────────
print(revenue['Jan'])          # label-based
print(revenue.iloc[0])         # position-based 
print(revenue['Jan':'Mar'])    # label slice — INCLUSIVE on both ends
print(revenue[revenue > 100_000])   # boolean mask
print(revenue.iloc[1:4])       # positional slice — exclusive end

120000
120000
Jan    120000
Feb     95000
Mar    142000
Name: Monthly Revenue, dtype: int64
Jan    120000
Mar    142000
May    175000
Name: Monthly Revenue, dtype: int64
Feb     95000
Mar    142000
Apr     88000
Name: Monthly Revenue, dtype: int64


mom_growth=>  
pct_change  
(current value - previous value) / previous value  
(Feb - Jan) / Jan  
(Mar - Feb) / Feb  
...
Then * 100 converts it into percentage.  
if Jan = 100, Feb = 120  
Then:(120 - 100) / 100 = 0.20 → 20%  
jan =120000, feb =95000  
pct_change=(120000-95000)/12000=-20%   

Jan
Jan + Feb
Jan + Feb + Mar
...

| Month | Revenue | Cumulative |
| ----- | ------- | ---------- |
| Jan   | 100     | 100        |
| Feb   | 200     | 300        |
| Mar   | 150     | 450        |

In [44]:
# ── 2.3 Vectorized Operations (why Pandas is fast) ────────────────────────────
# Apply a 10% growth projection — no Python loop needed
projected = revenue * 1.10 #[120000 * 1.10, 95000 * 1.10, ...]
print(projected)

#Month -over-groth growth %
#pct_change => % change between current  and prev value
# month over month growth
mom_growth = revenue.pct_change() * 100 ## (current - previous) / previous → then *100 for %
print('\nmom_growth %:')
print(mom_growth.round(2))

#cumulative revenue

print('\ncumulative revenue:')
print(revenue.cumsum())

Jan    132000.0
Feb    104500.0
Mar    156200.0
Apr     96800.0
May    192500.0
Name: Monthly Revenue, dtype: float64

mom_growth %:
Jan      NaN
Feb   -20.83
Mar    49.47
Apr   -38.03
May    98.86
Name: Monthly Revenue, dtype: float64

cumulative revenue:
Jan    120000
Feb    215000
Mar    357000
Apr    445000
May    620000
Name: Monthly Revenue, dtype: int64


In [48]:
# ── 2.4 Essential Series Methods (used every day in analytics) ─────────────────
#create a series of student scores, including a missing value (np.nan)
scores = pd.Series([88, 92, 75, 88, 95, 60, 72, 88, 91, np.nan])
print('--- Descriptive Statistics ---')
print(scores.describe())
print(f'\nMean   : {scores.mean():.2f}') # Avg
print(f'Median : {scores.median():.2f}') # middle value after soring
print(f'Std    : {scores.std():.2f}') #std deviation (spread of data around mean)
print(f'Mode   : {scores.mode()[0]}')# most frequent val, [0] picks the first mode if multiple exist

print('\n--- Value Counts (critical for categorical analysis) ---')
print(scores.value_counts()) # count how many times each unique value apprears

print('\n--- Null check ---')
print(f'Nulls: {scores.isnull().sum()} | Non-nulls: {scores.notna().sum()}') #count missing value (nan) and non missing val


--- Descriptive Statistics ---
count     9.000000
mean     83.222222
std      11.605794
min      60.000000
25%      75.000000
50%      88.000000
75%      91.000000
max      95.000000
dtype: float64

Mean   : 83.22
Median : 88.00
Std    : 11.61
Mode   : 88.0

--- Value Counts (critical for categorical analysis) ---
88.0    3
92.0    1
75.0    1
95.0    1
60.0    1
72.0    1
91.0    1
Name: count, dtype: int64

--- Null check ---
Nulls: 1 | Non-nulls: 9


In [57]:
# ── 2.5 String Series (very common with customer data) ─────────────────────────
customers = pd.Series(['  Alice Johnson ', 'BOB SMITH', 'charlie brown', 
                       'Diana PRINCE  ', 'eve  williams'])
                       # Clean names like you would in a real ETL pipeline
customers_clean = (customers
    .str.strip()          # remove leading/trailing whitespace
    .str.title()          # proper case #"JOHN DOE"=> "John Doe"
    .str.replace(r'\s+', ' ', regex=True)  # collapse multiple spaces,     # "eve  williams" -> "Eve Williams"
)
print(customers_clean)

# Extract first name
print('\nFirst Names:')
print(customers_clean.str.split().str[0])


0    Alice Johnson
1        Bob Smith
2    Charlie Brown
3     Diana Prince
4     Eve Williams
dtype: str

First Names:
0      Alice
1        Bob
2    Charlie
3      Diana
4        Eve
dtype: object


In [60]:
# ── 2.6 apply() — when vectorized methods aren't enough ───────────────────────
# apply() lets you run a custom Python function on each element of a Series
# BUT it is slower because it works row-by-row (Python-level loop hidden inside)

revenue_series = pd.Series([120_000, 95_000, 142_000, 88_000, 175_000])

# custum fun to classify revenue into bussiness category
def classify_revenue(val):
    if val >= 150_000: return 'High' # >=150K-> high segment
    elif val >= 100_000: return 'Medium' # 100K to 149K->medium segment
    else: return 'Low' # below 100K -> low segment

    # apply() way
print(revenue_series.apply(classify_revenue))





0    Medium
1       Low
2    Medium
3       Low
4      High
dtype: str


In [69]:
# BETTER way — pd.cut (vectorized, faster)
print(pd.cut(revenue_series,
             bins=[0, 100_000, 150_000, float('inf')],
             labels=['Low', 'Medium', 'High']))
#print(revenue.cumsum())



0    Medium
1       Low
2    Medium
3       Low
4      High
dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']


0 → 100k → Low  
100k → 150k → Medium  
150k → ∞ → High  

So if data is:  

| Value  | Category |
| ------ | -------- |
| 120000 | Medium   |
| 95000  | Low      |
| 175000 | High     |


Output becomes:  ['Medium', 'Low', 'High']  

So output:
0    Medium  
1    Low  
2    Medium  
3    Low  
4    High  

This is NOT numbers anymore  
👉 This is categorical data  

🧾 Comment you should write  
#### convert numeric revenue into categories using bins (vectorized, no loop)